# 12 — vLLM: LongBench Benchmark

This notebook evaluates KV cache compression on
[LongBench](https://github.com/THUDM/LongBench) tasks using
[vLLM](https://github.com/vllm-project/vllm) with Qwen3-8B.

We test two compression algorithms (**full_replacement**, **filtering**) at
multiple compression ratios on two LongBench tasks:
- **gov_report** — summarization (high decoding stress, long generated output)
- **hotpotqa** — multi-hop QA (multi-document reasoning)

Scoring uses HuggingFace `evaluate`:
- ROUGE-L for gov_report (summarization)
- F1 for hotpotqa (QA)

Results are saved to `results/vllm_longbench/` for comparison in later notebooks.

## Configuration

In [ ]:
MODEL_NAME = "Qwen/Qwen3-8B"

COMPRESSION_RATIOS = [0.01, 0.25, 0.50, 0.75]

FRACTION = 0.01

LONGBENCH_TASKS = ["gov_report", "hotpotqa"]

MAX_NEW_TOKENS = {
    "gov_report": 512,
    "hotpotqa": 64,
}

PRESS_CONFIGS = {
    "full_replacement": lambda cr: {
        "model": MODEL_NAME,
        "dtype": "auto",
        "gpu_memory_utilization": 0.90,
        "trust_remote_code": True,
        "attention_config": {"backend": "FLASH_ATTN"},
        "kv_compression_algorithm": "full_replacement",
        "kv_compression_ratio": cr,
        "enable_prefix_caching": False,
    },
    "filtering": lambda cr: {
        "model": MODEL_NAME,
        "dtype": "auto",
        "gpu_memory_utilization": 0.90,
        "trust_remote_code": True,
        "attention_config": {"backend": "FLASH_ATTN"},
        "kv_compression_algorithm": "filtering",
        "kv_compression_ratio": cr,
    },
}

In [ ]:
import sys
import builtins

_original_print = builtins.print

def print(*args, **kwargs):
    _original_print(*args, **kwargs)
    if sys.stdout is not sys.__stdout__:
        kwargs['file'] = sys.__stdout__
        kwargs['flush'] = True
        _original_print(*args, **kwargs)

In [ ]:
import sys
import os

FORK_DIR = "/opt/app-root/src/vllm-fork"

if os.path.isdir(FORK_DIR) and os.listdir(FORK_DIR):
    sys.path.insert(0, FORK_DIR)
    import vllm
    print(f"Using FORK vLLM (version: {vllm.__version__})")
else:
    import vllm
    print(f"Using SYSTEM vLLM (version: {vllm.__version__})")

In [ ]:
import gc
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU detected.")

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
allocated_gb = torch.cuda.memory_allocated() / 1e9
reserved_gb = torch.cuda.memory_reserved() / 1e9

print(f"GPU:        {torch.cuda.get_device_name(0)}")
print(f"VRAM:       {vram_gb:.1f} GB total")
print(f"Allocated:  {allocated_gb:.2f} GB")
print(f"Reserved:   {reserved_gb:.2f} GB")
print(f"Free:       {vram_gb - reserved_gb:.1f} GB (approx)")

if allocated_gb > 1.0:
    print(
        "\n\u26a0  GPU memory is not free — a model from another notebook may still be loaded.\n"
        "   Restart this kernel before proceeding."
    )

def cleanup_vllm(llm):
    llm.llm_engine.engine_core.shutdown()
    del llm
    gc.collect()
    torch.cuda.empty_cache()

## 1. Load LongBench Datasets

In [ ]:
from datasets import load_dataset

longbench_datasets = {}
for task_name in LONGBENCH_TASKS:
    ds = load_dataset("THUDM/LongBench", task_name, split="test")
    if FRACTION < 1.0:
        n = max(1, int(len(ds) * FRACTION))
        ds = ds.select(range(n))
    longbench_datasets[task_name] = ds
    print(f"{task_name}: {len(ds)} examples")

## 2. Load Scoring Metrics

In [ ]:
import evaluate

rouge_metric = evaluate.load("rouge")
squad_metric = evaluate.load("squad")

TASK_METRICS = {
    "gov_report": "rouge",
    "hotpotqa": "squad",
}

print("Loaded scoring metrics: rouge (gov_report), squad F1 (hotpotqa)")

## 3. Prepare Prompts

Apply the model's chat template to each LongBench example.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

prompt_configs = {}
max_context_tokens = 0

for task_name in LONGBENCH_TASKS:
    ds = longbench_datasets[task_name]
    task_prompts = []

    for row in ds:
        user_msg = row["context"] + "\n\n" + row["input"]
        messages = [{"role": "user", "content": user_msg}]
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
        )

        n_tokens = len(tokenizer.encode(prompt, add_special_tokens=False))
        max_context_tokens = max(max_context_tokens, n_tokens)

        task_prompts.append({
            "prompt": prompt,
            "task": task_name,
            "answers": row["answers"],
        })

    prompt_configs[task_name] = task_prompts
    print(f"{task_name}: {len(task_prompts)} prompts prepared")

print(f"\nMax context tokens: {max_context_tokens}")

## 4. Run Batch Inference

vLLM processes all prompts per task in a single batch via its internal
scheduler. A new LLM instance is created per (algorithm, compression_ratio)
combination.

In [ ]:
import time
from vllm import LLM, SamplingParams

def get_gpu_memory_used_gb() -> float:
    free, total = torch.cuda.mem_get_info()
    return (total - free) / 1e9

sampling_params_per_task = {
    task: SamplingParams(temperature=0.0, max_tokens=max_tok)
    for task, max_tok in MAX_NEW_TOKENS.items()
}

configs = [("no_press", 0.0)]
for press_name in PRESS_CONFIGS:
    for ratio in COMPRESSION_RATIOS:
        configs.append((press_name, ratio))

all_results = []

for press_name, ratio in configs:
    llm = None
    try:
        if press_name == "no_press":
            llm = LLM(
                model=MODEL_NAME,
                dtype="auto",
                gpu_memory_utilization=0.90,
                trust_remote_code=True,
                attention_config={"backend": "FLASH_ATTN"},
            )
        else:
            llm_kwargs = PRESS_CONFIGS[press_name](ratio)
            llm = LLM(**llm_kwargs)

        for task_name in LONGBENCH_TASKS:
            prompts_list = prompt_configs[task_name]
            prompts = [pc["prompt"] for pc in prompts_list]
            sp = sampling_params_per_task[task_name]

            label = f"{press_name} | ratio={ratio} | {task_name}"
            print(f"\n{'='*60}")
            print(f"Running: {label} ({len(prompts)} examples)")
            print(f"{'='*60}")

            mem_before = get_gpu_memory_used_gb()
            start = time.perf_counter()

            outputs = llm.generate(prompts, sp)

            batch_elapsed = time.perf_counter() - start
            mem_after = get_gpu_memory_used_gb()
            peak_mem = max(mem_before, mem_after)

            for i, output in enumerate(outputs):
                predicted_answer = output.outputs[0].text.strip()
                pc = prompts_list[i]

                all_results.append({
                    "framework": "vllm",
                    "press": press_name,
                    "compression_ratio": ratio,
                    "task": task_name,
                    "answers": pc["answers"],
                    "predicted_answer": predicted_answer,
                    "elapsed_sec": round(batch_elapsed / len(prompts), 3),
                    "peak_gpu_mem_gb": round(peak_mem, 3),
                })

            total_gen_tokens = sum(len(o.outputs[0].token_ids) for o in outputs)
            throughput = total_gen_tokens / batch_elapsed if batch_elapsed > 0 else 0
            print(f"  Done: {batch_elapsed:.1f}s — {throughput:.1f} tok/s — peak mem={peak_mem:.2f} GB")

    finally:
        if llm is not None:
            cleanup_vllm(llm)

print(f"\nTotal results: {len(all_results)}")

## 5. Score & Results

Score predictions using HuggingFace `evaluate`:
- ROUGE-L for gov_report (summarization)
- F1 for hotpotqa (QA, via SQuAD metric)

In [ ]:
import pandas as pd

df = pd.DataFrame(all_results)

all_metrics = {}
rows = []

for (press, ratio, task), group in df.groupby(
    ["press", "compression_ratio", "task"]
):
    preds = group["predicted_answer"].tolist()
    refs = group["answers"].tolist()
    key = f"{press}__{ratio}__{task}"

    if TASK_METRICS[task] == "rouge":
        result = rouge_metric.compute(
            predictions=preds,
            references=[r[0] if isinstance(r, list) else r for r in refs],
        )
        score = round(result["rougeL"] * 100, 2)
        metric_name = "rougeL"
    else:
        squad_preds = [{"id": str(i), "prediction_text": p} for i, p in enumerate(preds)]
        squad_refs = [{"id": str(i), "answers": {"text": r if isinstance(r, list) else [r], "answer_start": [0] * (len(r) if isinstance(r, list) else 1)}} for i, r in enumerate(refs)]
        result = squad_metric.compute(predictions=squad_preds, references=squad_refs)
        score = round(result["f1"], 2)
        metric_name = "f1"

    all_metrics[key] = {metric_name: score}
    rows.append({
        "press": press, "compression_ratio": ratio,
        "task": task, "metric": metric_name, "score": score,
        "mean_time": round(group["elapsed_sec"].mean(), 3),
    })

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))

## 6. Save Results

In [ ]:
import json

os.makedirs("results/vllm_longbench", exist_ok=True)

predictions_path = "results/vllm_longbench/predictions.csv"
df.to_csv(predictions_path, index=False)
print(f"Saved predictions to {predictions_path}")

metrics_path = "results/vllm_longbench/metrics.json"
with open(metrics_path, "w") as f:
    json.dump(all_metrics, f, indent=2)
print(f"Saved metrics to {metrics_path}")